# Notebook 02 — Modèle TF-IDF

**Milestone M-02 | Étudiant B — IR Specialist**

Ce notebook couvre :
1. Indexation TF-IDF avec `TfidfVectorizer`
2. Fonction de recherche `search(query, k)` basée sur la similarité cosinus
3. Évaluation : Recall@10, Precision@10, MRR

## 1. Imports

In [ ]:
import json
import os
import pickle

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR    = '../data'
OUTPUTS_DIR = '../outputs'
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print('Imports OK')

## 2. Chargement des Données Préprocessées

In [ ]:
# Chargement du DataFrame produit par 01_eda.ipynb
df_docs = pd.read_pickle(os.path.join(DATA_DIR, 'df_docs_preprocessed.pkl'))

with open(os.path.join(DATA_DIR, 'queries_train.json'), 'r', encoding='utf-8') as f:
    queries_train = json.load(f)

with open(os.path.join(DATA_DIR, 'qgts_train.json'), 'r', encoding='utf-8') as f:
    qgts_train = json.load(f)  # dict : query_id -> {total_relevant_docs, relevant_doc_ids}

with open(os.path.join(DATA_DIR, 'queries_test.json'), 'r', encoding='utf-8') as f:
    queries_test = json.load(f)

# Mapping id -> index numérique
id_to_idx = {doc_id: idx for idx, doc_id in enumerate(df_docs['id'])}
idx_to_id = {idx: doc_id for doc_id, idx in id_to_idx.items()}

corpus = df_docs['content_clean'].tolist()

def get_relevant_ids(qgts, query_id):
    """Extrait les doc_ids pertinents depuis le ground truth."""
    if query_id not in qgts:
        return []
    return [e['doc_id'] for e in qgts[query_id]['relevant_doc_ids']]

def build_query_text(q):
    """Construit le texte de recherche depuis une entrée de requête."""
    parts = [q.get('text', ''), q.get('title', '')]
    if q.get('tags'):
        parts.append(' '.join(q['tags']))
    return ' '.join(p for p in parts if p).strip()

print(f'Corpus chargé          : {len(corpus):,} documents')
print(f'Requêtes train         : {len(queries_train)}')
print(f'Requêtes test          : {len(queries_test)}')

## 3. Indexation TF-IDF

In [ ]:
# Construction de l'index TF-IDF
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),   # unigrammes + bigrammes
    min_df=1,
    max_df=0.95,
    sublinear_tf=True     # log(TF) — réduit l'impact des termes très fréquents
)

tfidf_matrix = vectorizer.fit_transform(corpus)  # (n_docs, n_features)

print(f'Matrice TF-IDF : {tfidf_matrix.shape}')
print(f'Nombre de features (termes) : {len(vectorizer.get_feature_names_out())}')

## 4. Fonction de Recherche

In [ ]:
def search(query: str, k: int = 10) -> dict:
    """
    Recherche TF-IDF par similarité cosinus.

    Args:
        query: Requête en langage naturel.
        k: Nombre de résultats à retourner.

    Returns:
        dict avec 'topk_indices' (indices dans df_docs) et 'topk_scores'.
    """
    query_vec = vectorizer.transform([query])
    scores    = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_k_idx = np.argsort(scores)[::-1][:k]
    return {
        'topk_indices': top_k_idx.tolist(),
        'topk_scores':  scores[top_k_idx].tolist()
    }

# Test rapide
result = search('neural networks deep learning', k=5)
print('Top 5 résultats pour "neural networks deep learning" :')
for idx, score in zip(result['topk_indices'], result['topk_scores']):
    print(f"  [{score:.4f}] {df_docs.iloc[idx]['id']} — {df_docs.iloc[idx]['title']}")

## 5. Métriques d'Évaluation

In [ ]:
def recall_at_k(retrieved_indices, relevant_ids, k, id_to_idx):
    """Recall@k : fraction des documents pertinents retrouvés dans le top-k."""
    relevant_set = set(id_to_idx[rid] for rid in relevant_ids if rid in id_to_idx)
    retrieved_set = set(retrieved_indices[:k])
    if not relevant_set:
        return 0.0
    return len(retrieved_set & relevant_set) / len(relevant_set)


def precision_at_k(retrieved_indices, relevant_ids, k, id_to_idx):
    """Precision@k : fraction du top-k qui est pertinente."""
    relevant_set = set(id_to_idx[rid] for rid in relevant_ids if rid in id_to_idx)
    retrieved_set = set(retrieved_indices[:k])
    return len(retrieved_set & relevant_set) / k


def mrr(retrieved_indices, relevant_ids, id_to_idx):
    """Mean Reciprocal Rank : 1/rang du premier document pertinent trouvé."""
    relevant_set = set(id_to_idx[rid] for rid in relevant_ids if rid in id_to_idx)
    for rank, idx in enumerate(retrieved_indices, start=1):
        if idx in relevant_set:
            return 1.0 / rank
    return 0.0

print('Fonctions de métriques définies : recall_at_k, precision_at_k, mrr')

## 6. Évaluation sur les Requêtes d'Entraînement

In [ ]:
K = 10
recalls, precisions, mrrs_list = [], [], []

for query_entry in queries_train:
    query_id     = query_entry['id']
    query_text   = build_query_text(query_entry)
    relevant_ids = get_relevant_ids(qgts_train, query_id)

    if not relevant_ids:
        continue

    result    = search(query_text, k=K)
    retrieved = result['topk_indices']

    recalls.append(recall_at_k(retrieved, relevant_ids, K, id_to_idx))
    precisions.append(precision_at_k(retrieved, relevant_ids, K, id_to_idx))
    mrrs_list.append(mrr(retrieved, relevant_ids, id_to_idx))

tfidf_results = {
    'model':           'TF-IDF',
    f'Recall@{K}':     np.mean(recalls),
    f'Precision@{K}':  np.mean(precisions),
    'MRR':             np.mean(mrrs_list)
}

print(f'Évaluation sur {len(recalls)} requêtes (avec ground truth)')
print('=== Résultats TF-IDF ===')
for key, val in tfidf_results.items():
    if isinstance(val, float):
        print(f'{key:15s}: {val:.4f}')
    else:
        print(f'{key:15s}: {val}')

In [ ]:
# Sauvegarde des résultats pour le notebook de consolidation
with open(os.path.join(OUTPUTS_DIR, 'tfidf_results.pkl'), 'wb') as f:
    pickle.dump(tfidf_results, f)

# Sauvegarde du vectorizer pour réutilisation (classifieur, etc.)
with open(os.path.join('../models', 'tfidf_vectorizer.pkl'), 'wb') as f:
    pickle.dump(vectorizer, f)
with open(os.path.join('../models', 'tfidf_matrix.pkl'), 'wb') as f:
    pickle.dump(tfidf_matrix, f)

print('Résultats et modèle sauvegardés.')

## 7. Analyse Qualitative

In [ ]:
# Analyse sur 3 requêtes d'entraînement
sample_queries = queries_train[:3]

for q_entry in sample_queries:
    q_text   = build_query_text(q_entry)
    q_id     = q_entry['id']
    relevant = get_relevant_ids(qgts_train, q_id)
    result   = search(q_text, k=5)

    print(f'Requête : "{q_text[:80]}"')
    print(f'Docs pertinents ({len(relevant)}) : {relevant[:3]}...')
    print('Top-5 TF-IDF :')
    for idx, score in zip(result['topk_indices'], result['topk_scores']):
        doc_id = idx_to_id[idx]
        marker = ' ✓' if doc_id in relevant else ''
        print(f"  [{score:.4f}] {doc_id} — {df_docs.iloc[idx]['title'][:60]}{marker}")
    print()

## 8. Résumé

Le modèle TF-IDF :
- Utilise `TfidfVectorizer` avec unigrammes et bigrammes, `sublinear_tf=True`
- Recherche par similarité cosinus entre la requête et la matrice de documents
- Interface standardisée : `search(query, k) -> {'topk_indices': [...], 'topk_scores': [...]}`

Les résultats sont sauvegardés dans `outputs/tfidf_results.pkl`.